In [1]:
import torch
print("Có GPU không? :", torch.cuda.is_available())  # Phải trả về True
print("Phiên bản CUDA trong PyTorch:", torch.version.cuda)
print("Tên GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Không có GPU")

# Kiểm tra GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Đang sử dụng: {device}") # phải là: Đang sử dụng: cuda

Có GPU không? : True
Phiên bản CUDA trong PyTorch: 11.8
Tên GPU: NVIDIA GeForce GTX 1650
Đang sử dụng: cuda


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, transforms, datasets
from torch.utils.data import DataLoader
from PIL import Image
import numpy as np
import os
import ssl

from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

In [3]:
# Kiểm tra GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Đang sử dụng: {device}")

Đang sử dụng: cuda


In [ ]:
# Sử dụng khi thuê GPU trên máy ảo để tải base model
# ssl._create_default_https_context = ssl._create_unverified_context

## train model

In [4]:
# Load mô hình ResNet50 đã pretrain
base_model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
num_ftrs = base_model.fc.in_features

In [5]:
# Thêm các lớp fully connected (số nhãn của dataset)
num_classes = 9
base_model.fc = nn.Sequential(
    nn.Linear(num_ftrs, 1024),
    nn.ReLU(),
    nn.Linear(1024, num_classes),
    nn.Softmax(dim=1)
)

In [6]:
# Chuyển model sang GPU
base_model = base_model.to(device)

In [7]:
# Đóng băng các layer của ResNet50
for param in base_model.parameters():
    param.requires_grad = False

In [8]:
# Chỉ fine-tune 4 lớp cuối
for param in list(base_model.parameters())[-4:]:
    param.requires_grad = True

In [9]:
# Optimizer và loss function
optimizer = optim.Adam(base_model.fc.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

In [ ]:
# Chuẩn bị dữ liệu
data_dir = 'cic_ddos_2019_images'
train_dir = f"data/{data_dir}/train"
val_dir = f"data/{data_dir}/valid"
test_dir = f"data/{data_dir}/test"

In [11]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

In [12]:
train_dataset = datasets.ImageFolder(train_dir, transform=transform)
val_dataset = datasets.ImageFolder(val_dir, transform=transform)
test_dataset = datasets.ImageFolder(test_dir, transform=transform)


train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [13]:
# Training loop
def train(model, train_loader, val_loader, epochs=10):
    model.train()
    for epoch in range(epochs):
        running_loss = 0.0
        correct = 0
        total = 0
        
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)
        
        acc = 100 * correct / total
        print(f"Epoch {epoch+1}, Loss: {running_loss/len(train_loader):.4f}, Accuracy: {acc:.2f}%")

In [14]:
# Huấn luyện mô hình
train(base_model, train_loader, val_loader, epochs=20)


Epoch 1, Loss: 2.1783, Accuracy: 18.52%
Epoch 2, Loss: 2.0535, Accuracy: 44.44%
Epoch 3, Loss: 1.8942, Accuracy: 61.11%
Epoch 4, Loss: 1.7739, Accuracy: 68.52%
Epoch 5, Loss: 1.6929, Accuracy: 81.48%
Epoch 6, Loss: 1.5824, Accuracy: 87.04%
Epoch 7, Loss: 1.5450, Accuracy: 94.44%
Epoch 8, Loss: 1.4967, Accuracy: 96.30%
Epoch 9, Loss: 1.4903, Accuracy: 94.44%
Epoch 10, Loss: 1.4537, Accuracy: 96.30%
Epoch 11, Loss: 1.4198, Accuracy: 98.15%
Epoch 12, Loss: 1.4112, Accuracy: 98.15%
Epoch 13, Loss: 1.4124, Accuracy: 98.15%
Epoch 14, Loss: 1.3974, Accuracy: 100.00%
Epoch 15, Loss: 1.3948, Accuracy: 100.00%
Epoch 16, Loss: 1.4002, Accuracy: 100.00%
Epoch 17, Loss: 1.3776, Accuracy: 100.00%
Epoch 18, Loss: 1.3783, Accuracy: 100.00%
Epoch 19, Loss: 1.3812, Accuracy: 100.00%
Epoch 20, Loss: 1.3772, Accuracy: 100.00%


In [ ]:
# Fine-tune 4 lớp cuối
train(base_model, train_loader, val_loader, epochs=5)

Epoch 1, Loss: 1.5792, Accuracy: 85.19%
Epoch 2, Loss: 1.5584, Accuracy: 85.19%


In [ ]:
# Lưu mô hình
torch.save(base_model.state_dict(), "models/fine_tuned_model.pth")

## Load model và dự đoán

In [13]:
# lựa chọn thiết bị để tải model và tải dữ liệu dự đoán
# predict_device = torch.device("cuda" if torch.cuda.is_available() else "cpu") # gpu
predict_device = torch.device("cpu") # cpu

In [ ]:
num_classes = 9

# Load mô hình ResNet50 đã lưu
tuned_model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)  # Khởi tạo mô hình
num_ftrs = tuned_model.fc.in_features
tuned_model.fc = nn.Sequential(
    nn.Linear(num_ftrs, 1024),
    nn.ReLU(),
    nn.Linear(1024, num_classes),
    nn.Softmax(dim=1)
)
tuned_model.load_state_dict(torch.load("models/resnet50.pth", map_location=predict_device))  # Load trọng số
tuned_model.to(predict_device)  # Chuyển model lên GPU
tuned_model.eval()  # Chuyển mô hình sang chế độ đánh giá (inference)


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 

In [15]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

In [ ]:
# Dự đoán trên một ảnh đơn
img_path = "data/cic_ddos_2019_images/test/Portmap/2.png"
img = Image.open(img_path)
img = transform(img).unsqueeze(0).to(predict_device)

with torch.no_grad():
    output = tuned_model(img)
    predicted_class = torch.argmax(output, dim=1).item()
    print(f"Dự đoán nhãn: {predicted_class}")

Dự đoán nhãn: 4


In [ ]:
test_dir = "data/cic_ddos_2019_images/test"
test_dataset = datasets.ImageFolder(test_dir, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)
criterion = nn.CrossEntropyLoss()


#### Tính toán Accuracy

In [ ]:

# Đánh giá trên tập test
def evaluate(model, test_loader):
    model.eval()
    correct = 0
    total = 0
    test_loss = 0.0
    
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(predict_device), labels.to(predict_device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            test_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)
    
    acc = 100 * correct / total
    print(f"Test Loss: {test_loss/len(test_loader):.4f}")
    print(f"Test Accuracy: {acc:.2f}%")

evaluate(tuned_model, test_loader)

#### tính toán F1-Score, Precision, Recall

In [18]:
class_names = test_dataset.classes

# Chạy dự đoán
y_true = []
y_pred = []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(predict_device), labels.to(predict_device)  # Chuyển dữ liệu về GPU
        outputs = tuned_model(images)
        _, preds = torch.max(outputs, 1)

        y_true.extend(labels.numpy())
        y_pred.extend(preds.numpy())

# In báo cáo kết quả
print(classification_report(y_true, y_pred, target_names=class_names))

              precision    recall  f1-score   support

      BENIGN       0.85      1.00      0.92       400
        LDAP       1.00      1.00      1.00       400
       MSSQL       1.00      1.00      1.00       400
     NetBIOS       0.98      1.00      0.99       400
     Portmap       1.00      1.00      1.00       400
         Syn       1.00      1.00      1.00       400
         UDP       1.00      1.00      1.00       400
      UDPLag       0.97      0.99      0.98       400
     WebDDoS       0.00      0.00      0.00        88

    accuracy                           0.97      3288
   macro avg       0.87      0.89      0.88      3288
weighted avg       0.95      0.97      0.96      3288



f:\NCKH\code\DN\IDS-RESNET-50-System\train_model\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
f:\NCKH\code\DN\IDS-RESNET-50-System\train_model\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
f:\NCKH\code\DN\IDS-RESNET-50-System\train_model\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mo